In [ ]:
# Run first. Anchor the working directory to the project root so that
# relative paths like "data/processed/..." resolve no matter where the
# notebook is launched from (notebooks live in notebooks/, data at root).
# Idempotent: re-running keeps you at the root.
import os
from pathlib import Path

_root = Path.cwd()
while not (_root / "CLAUDE.md").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
print("Working directory:", Path.cwd())

# Importing libraries

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# Load and Audit

## Beauty table
All_Beauty: only 0.26% of users have 5+ reviews → unusable for repurchase analysis, kept as pipeline sandbox.

In [3]:
from datasets import load_dataset

reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    split="full",
    trust_remote_code=True,
)
reviews.to_pandas().to_parquet("data/raw/beauty_reviews.parquet")

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True,
)
meta.to_pandas().to_parquet("data/raw/beauty_meta.parquet")

raw/meta_categories/meta_All_Beauty.json(…): reconstructing file:   0%|          |  0.00B /  213MB            

raw/meta_categories/meta_All_Beauty.json(…): downloading bytes:           |  0.00B            

Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

In [ ]:
df_beauty = pd.read_parquet("data/raw/beauty_reviews.parquet")
print(df_beauty.shape)
print(df_beauty.columns.tolist())
df_beauty.head()

(701528, 10)
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True


In [ ]:
counts = df_beauty.groupby("user_id").size()

print("unique users:", counts.shape[0])
print("median reviews per user:", counts.median())
print("share of users with 5+ reviews:", (counts >= 5).mean())
print(counts.value_counts().head(10))

unique users: 631986
median reviews per user: 1.0
share of users with 5+ reviews: 0.0025633479222641007
1     583553
2      39274
3       5713
4       1826
5        558
6        351
7        181
8        130
9         70
11        35
Name: count, dtype: int64


## Grocery Table

In [2]:
from datasets import load_dataset

grocery = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Grocery_and_Gourmet_Food",
    split="full",
    trust_remote_code=True,
)
grocery.to_pandas().to_parquet("data/raw/grocery_reviews.parquet")

In [ ]:
df_grocery = pd.read_parquet("data/raw/grocery_reviews.parquet")
print(df_grocery.shape)
print(df_grocery.columns.tolist())
df_grocery.head()

(14318520, 10)
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Excellent! Yummy!,Excellent!! Yummy! Great with other foods and...,[],B00CM36GAQ,B00CM36GAQ,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587854482395,0,True
1,5.0,Delicious!!! Yum!,Excellent! The best! I use it with my beef a...,[],B074J5WVYH,B0759B7KLH,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587854400380,0,True
2,5.0,"Extremely Delicious, but expensive imo",These are very tasty. They are extremely soft ...,[],B079TRNVHX,B079TRNVHX,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587853224527,1,True
3,5.0,Delicious!,My favorite!,[],B07194LN2Z,B07194LN2Z,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1581313319614,0,True
4,5.0,Great taste,Great for making brownies and crinkle cookies.,[],B005CD4196,B005CD4196,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1581313294965,7,True


 ### Filter to repeat users and sample

In [ ]:
df_grocery = pd.read_parquet("data/raw/grocery_reviews.parquet")
counts_g = df_grocery.groupby("user_id").size()

print("total reviews:", len(df_grocery))
print("unique users:", counts_g.shape[0])
print("median reviews per user:", counts_g.median())
print("share of users with 5+ reviews:", (counts_g >= 5).mean())
print("users with 5+ reviews:", (counts_g >= 5).sum())

total reviews: 14318520
unique users: 7034393
median reviews per user: 1.0
share of users with 5+ reviews: 0.07026448479634277
users with 5+ reviews: 494268


In [ ]:
repeat_users = counts_g[counts_g >= 5].index
df_repeat = df_grocery[df_grocery["user_id"].isin(repeat_users)]

print("reviews in repeat-user set:", len(df_repeat))
print("users:", df_repeat["user_id"].nunique())
print("mean reviews per user:", len(df_repeat) / df_repeat["user_id"].nunique())

reviews in repeat-user set: 4892214
users: 494268
mean reviews per user: 9.897897496904513


In [10]:
rng = np.random.default_rng(42)
sampled_users = rng.choice(repeat_users, size=40_000, replace=False)

df_sample = df_repeat[df_repeat["user_id"].isin(sampled_users)]

print("reviews:", len(df_sample))
print("users:", df_sample["user_id"].nunique())
print("mean reviews per user:", len(df_sample) / df_sample["user_id"].nunique())

df_sample.to_parquet("data/interim/grocery_sample.parquet")

reviews: 393434
users: 40000
mean reviews per user: 9.83585


# Build the modeling table

In [4]:
df = pd.read_parquet("data/interim/grocery_sample.parquet")

# 1. timestamp: milliseconds since 1970 -> real dates
df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

# 2. text: title and body are separate fields; the title often carries
#    the sharpest sentiment ("Terrible!"), so keep both
df["full_text"] = (df["title"].fillna("") + ". " + df["text"].fillna("")).str.strip()

# 3. labels: 1-2 = negative (0), 4-5 = positive (1), 3 = dropped
df["label"] = df["rating"].map({1: 0, 2: 0, 4: 1, 5: 1})

# 4. keep only usable rows
before = len(df)
df_model = df[df["label"].notna()]                        # drops the 3s
df_model = df_model[df_model["full_text"].str.len() >= 20].copy()  # drops "Good." etc.
df_model["label"] = df_model["label"].astype(int)

print(f"started with: {before}")
print(f"after dropping 3-star: {(df['label'].notna()).sum()}")
print(f"after dropping short text: {len(df_model)}")
print()
print(df_model["label"].value_counts())
print(df_model["label"].value_counts(normalize=True))

started with: 393434
after dropping 3-star: 365503
after dropping short text: 352253

label
1    304010
0     48243
Name: count, dtype: int64
label
1    0.863044
0    0.136956
Name: proportion, dtype: float64


In [5]:
df_model.to_parquet("data/processed/modeling_table.parquet")
print("saved:", len(df_model), "rows")

saved: 352253 rows


In [9]:
import pandas as pd

df_sam = pd.read_parquet("data/interim/grocery_sample.parquet")

# Share of reviews that are verified purchases
print(df_sam["verified_purchase"].value_counts())
print(df_sam["verified_purchase"].value_counts(normalize=True))

# Does verification relate to rating? (unverified reviews are sometimes
# incentivized or fake, which tends to skew positive)
print(df_sam.groupby("verified_purchase")["rating"].mean())

verified_purchase
True     353305
False     40129
Name: count, dtype: int64
verified_purchase
True     0.898003
False    0.101997
Name: proportion, dtype: float64
verified_purchase
False    4.101672
True     4.317946
Name: rating, dtype: float64


In [ ]:
df_cust = pd.read_parquet("data/interim/grocery_sample.parquet")
df_cust = df_cust[df_cust["verified_purchase"] == True].copy()

df_cust["date"] = pd.to_datetime(df_cust["timestamp"], unit="ms")
df_cust = df_cust.sort_values(["user_id", "date"])

df_cust["next_date"] = df_cust.groupby("user_id")["date"].shift(-1)
df_cust["days_to_next"] = (df_cust["next_date"] - df_cust["date"]).dt.days

dataset_end = df_cust["date"].max()
df_cust["observable"] = (dataset_end - df_cust["date"]).dt.days >= 365

df_cust["repurchased_12m"] = None
df_cust.loc[df_cust["observable"], "repurchased_12m"] = (
    df_cust.loc[df_cust["observable"], "days_to_next"] <= 365
)

df_cust.to_parquet("data/processed/customer_reviews.parquet")

print("dataset ends:", dataset_end.date())
print("verified reviews:", len(df_cust))
print("observable:", df_cust["observable"].sum())
print(df_cust["repurchased_12m"].value_counts(dropna=False))

dataset ends: 2023-09-04
verified reviews: 353305
observable: 323238
repurchased_12m
True     247556
False     75682
None      30067
Name: count, dtype: int64


In [ ]:
obs = df_cust[df_cust["observable"] == True].copy()
obs["repurchased_12m"] = obs["repurchased_12m"].astype(bool)
obs["sentiment"] = obs["rating"].map({1: "negative", 2: "negative",
                                      3: "neutral",
                                      4: "positive", 5: "positive"})

by_sentiment = obs.groupby("sentiment")["repurchased_12m"].agg(["mean", "count"])
by_rating = obs.groupby("rating")["repurchased_12m"].agg(["mean", "count"])

print(by_sentiment, "\n")
print(by_rating)

gap = by_sentiment.loc["positive", "mean"] - by_sentiment.loc["negative", "mean"]
print(f"\nrepurchase gap (positive - negative): {gap:.3f}")

               mean   count
sentiment                  
negative   0.734014   38487
neutral    0.768410   22203
positive   0.770316  262548 

            mean   count
rating                  
1.0     0.726305   24341
2.0     0.747278   14146
3.0     0.768410   22203
4.0     0.788324   33386
5.0     0.767693  229162

repurchase gap (positive - negative): 0.036
